# Analisis Kemiskinan Kabupaten/Kota

Notebook ini memuat proses kualitas data, pembersihan, EDA, korelasi, model klasifikasi eksploratif, dan ringkasan temuan.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('../data/raw/Dataset_Kemiskinan.csv', sep=';', decimal=',')
df.shape

## 1. Pemeriksaan kualitas data
Data diperiksa untuk duplikat, nilai kosong, dan nilai di luar rentang wajar.

In [ ]:
print('Baris kembar persis:', df.duplicated().sum())
print('Nilai kosong:')
print(df.isna().sum())
df.describe(include='all').T

## 2. Pembersihan data

In [ ]:
batas = {
    'Kemiskinan': (0, 60), 'RLS': (3, 13),
    'Pengeluaran': (4000, 30000), 'IPM': (45, 90),
    'UHH': (55, 80), 'Sanitasi': (5, 100),
    'AirMinum': (5, 100), 'TPT': (0, 15),
    'TPAK': (50, 100), 'PDRB': (1, np.inf)
}
df = df.drop_duplicates().reset_index(drop=True)
anomali = pd.DataFrame({c: df[c].notna() & ~df[c].between(*r)
                         for c, r in batas.items()})
print(anomali.sum().sort_values(ascending=False))
print('Baris dengan minimal satu anomali:', anomali.any(axis=1).sum())

bersih = df.copy()
for c, r in batas.items():
    bersih.loc[~bersih[c].between(*r), c] = np.nan
for c in batas:
    bersih[c] = bersih[c].fillna(bersih.groupby('Provinsi')[c].transform('median'))
    bersih[c] = bersih[c].fillna(bersih[c].median())

Kolom `Klasifikasi Kemiskinan` lama tidak digunakan karena tidak konsisten dengan nilai kemiskinan.

In [ ]:
fitur = list(batas)
print(bersih[fitur].describe().round(2))
bersih.drop(columns=['Klasifikasi Kemiskinan']).to_csv('../data/clean/data_bersih.csv', index=False)

## 3. Analisis eksplorasi

In [ ]:
prov = (bersih.groupby('Provinsi')[['Kemiskinan','IPM','Pengeluaran','TPT']]
         .mean().round(2).sort_values('Kemiskinan', ascending=False))
display(prov.head(10))
px.histogram(bersih, x='Kemiskinan', nbins=30,
             title='Sebaran persentase kemiskinan').show()
px.box(bersih, x='Provinsi', y='Kemiskinan',
       title='Kemiskinan per provinsi').update_xaxes(tickangle=60).show()

## 4. Korelasi Spearman dan heatmap
Heatmap berikut merangkum arah dan kekuatan hubungan monoton antarvariabel. Nilai mendekati -1 menunjukkan hubungan berlawanan, nilai mendekati +1 menunjukkan hubungan searah, dan nilai dekat 0 menunjukkan hubungan monoton yang lemah.

In [ ]:
corr = bersih[fitur].corr(method='spearman').round(2)
heatmap = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    color_continuous_midpoint=0,
    title='Heatmap Korelasi Spearman antarindikator',
    labels={'x': 'Indikator', 'y': 'Indikator', 'color': 'Spearman ρ'},
)
heatmap.update_layout(height=720, width=900, xaxis_tickangle=-35)
heatmap.show()

print('Korelasi terhadap kemiskinan:')
display(corr['Kemiskinan'].drop('Kemiskinan').sort_values().to_frame('Spearman ρ'))
px.scatter(bersih, x='Pengeluaran', y='Kemiskinan', color='Provinsi',
           hover_name='KabKota', trendline='ols',
           title='Pengeluaran dan kemiskinan').show()

## 5. Model klasifikasi eksploratif
Kuartil tertinggi kemiskinan dijadikan kelas tinggi. Kolom `Kemiskinan` dikeluarkan dari fitur agar tidak terjadi kebocoran label.

In [ ]:
bersih['Tinggi'] = (bersih['Kemiskinan'] >= bersih['Kemiskinan'].quantile(.75)).astype(int)
X = bersih[[c for c in fitur if c != 'Kemiskinan']]
y = bersih['Tinggi']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
model.fit(Xtr, ytr)
print(classification_report(yte, model.predict(Xte), digits=3))
penting = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
px.bar(penting, title='Variabel paling berpengaruh pada model').show()

## 6. Batasan
Hasil ini bersifat deskriptif dan model hanya untuk eksplorasi; korelasi bukan sebab-akibat dan imputasi tidak menggantikan verifikasi sumber.